# Dam Structural Health Monitoring
## VT Crack Quantification ROIs Dataset → CNN → RUL

---

### Dataset structure (what you downloaded)
```
dataset/
├── Cropped/
│   ├── Sample_1/
│   │   ├── Sample_1_0.5_cropped.JPG
│   │   ├── Sample_1_0.75_cropped.JPG
│   │   └── Sample_1_1_cropped.JPG
│   ├── Sample_2/ ... Sample_15/
├── Ground_Truth_Masks/
│   ├── Sample_1/
│   │   ├── Sample_1_0.5_cropped.PNG
│   │   ├── Sample_1_0.75_cropped.PNG
│   │   └── Sample_1_1_cropped.PNG
│   └── ...
├── ROIs/                        (ROI binary masks — not used for CNN)
└── ROI_Measurements.csv         ← ground-truth crack widths (mm)
```

### What the CSV contains
- **15 physical cracks × 3 standoff distances** = **45 images**
- **5 manually-measured ROI widths per image** (mm, ruler-calibrated)
- Columns: `Sample, Resolution, ROI, X, Y, Crack Ruler Measurement`

### What the CNN predicts (regression targets per image)
| Target | Source | Unit |
|--------|--------|------|
| `mean_width_mm` | mean of 5 ROI measurements | mm |
| `max_width_mm` | max of 5 ROI measurements | mm |
| `crack_length_px` | skeleton of Ground Truth Mask | px |
| `orientation_deg` | PCA of Ground Truth Mask | ° |

### Why this design?
- Width is **directly labelled** in the CSV (ruler-calibrated) — highly accurate
- Length and orientation are **computed from the binary masks** — standard CV approach
- 45 images is small → we use **heavy augmentation + frozen ResNet50 backbone**

### Full pipeline
```
ROI_Measurements.csv + Ground_Truth_Masks/
        ↓  Section 1
  labels.csv  (one row per image, 4 targets)
        ↓  Section 2-4
  CNN training  (ResNet50 backbone)
        ↓  Section 5
  Inference on new image → mean_width, max_width, length, orientation
        ↓  Section 6-7
  LEFM + Paris' Law → K_I/K_IC ratio → RUL (years)
```

## 0 · Imports & Setup

In [ ]:
# ── Install (run once, then restart kernel) ───────────────────────────────────
# !pip install torch torchvision opencv-python-headless pillow pandas numpy matplotlib scikit-learn

In [ ]:
import os, sys, math, time, csv, json, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torchvision.models import ResNet50_Weights
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

C_BLUE, C_GREEN, C_ORANGE, C_RED, C_PURPLE, C_GREY = \
    '#00C8FF', '#39FF14', '#FF6B35', '#FF2D55', '#BF5FFF', '#8E9BB0'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

## 1 · Configuration  

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SET THESE PATHS TO MATCH YOUR DOWNLOADED DATASET FOLDER
# ═══════════════════════════════════════════════════════════════════════════════

DATASET_ROOT  = './dataset'          # folder containing Cropped/, Ground_Truth_Masks/, ROI_Measurements.csv
CSV_PATH      = f'{DATASET_ROOT}/ROI_Measurements.csv'
IMAGES_DIR    = f'{DATASET_ROOT}/Cropped'
MASKS_DIR     = f'{DATASET_ROOT}/Ground_Truth_Masks'
SAVE_DIR      = './checkpoints'
LABELS_CSV    = f'{SAVE_DIR}/labels.csv'    # generated in Section 1
BEST_MODEL    = f'{SAVE_DIR}/best_model.pt'

# ── Training ──────────────────────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 8       # small: only 45 images total
EPOCHS       = 80
LR           = 5e-5
PATIENCE     = 20
NUM_WORKERS  = 2
VAL_SPLIT    = 0.15    # fraction for validation  (≈7 images)
TEST_SPLIT   = 0.15    # fraction for test        (≈7 images)
RANDOM_SEED  = 42

# ── Dam physical parameters ───────────────────────────────────────────────────
DAM_H   = 100.0   # dam height (m)
DAM_T   = 12.0    # thickness at crack location (m)
KIC     = 1.0     # fracture toughness (MPa·√m)  — concrete: 0.7–1.5
Y_GEOM  = 1.1     # geometry factor              — surface crack: 1.0–1.2
PARIS_C = 1e-12   # Paris constant  m/(MPa·√m)^m·s
PARIS_M = 2.5     # Paris exponent

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print('Configuration loaded ✓')

## 2 · Build `labels.csv` from CSV + Masks

**Combines two label sources:**
- **Width** (mean & max mm) → directly from `ROI_Measurements.csv` (ruler-calibrated, highly accurate)
- **Length & Orientation** → computed from `Ground_Truth_Masks/` binary masks via skeletonisation + PCA

In [ ]:
# ── Helper: extract length + orientation from a binary mask ───────────────────

def mask_to_geometry(mask_path: str) -> dict:
    """
    Given a binary PNG mask, returns:
      length_px   : skeleton pixel count (proxy for crack length)
      orientation : principal axis angle in degrees (−90 to +90)
      area_px     : total crack area in pixels
      avg_width_px: area / length  (alternative width estimate)
    """
    img = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return {'length_px': 0, 'orientation_deg': 0.0,
                'area_px': 0, 'avg_width_px': 0.0, 'mask_found': False}

    # Binarise (handle both 0/1 and 0/255 masks)
    _, bw = cv2.threshold(img, 10, 255, cv2.THRESH_BINARY)

    if bw.max() == 0:
        return {'length_px': 0, 'orientation_deg': 0.0,
                'area_px': 0, 'avg_width_px': 0.0, 'mask_found': True}

    # ── Skeleton via distance transform ──────────────────────────────────────
    dist = cv2.distanceTransform(bw, cv2.DIST_L2, 5)
    _, skel = cv2.threshold(dist, 0.5 * dist.max(), 255, 0)
    skel_px = int(np.count_nonzero(skel))

    # ── Area ─────────────────────────────────────────────────────────────────
    area_px = int(np.count_nonzero(bw))
    avg_w   = float(area_px / skel_px) if skel_px > 0 else 0.0

    # ── Orientation via PCA on all crack pixels ───────────────────────────────
    ys, xs = np.where(bw > 0)
    pts = np.column_stack([xs, ys]).astype(np.float32)
    orient_deg = 0.0
    if len(pts) >= 5:
        _, eigvecs = cv2.PCACompute(pts, mean=np.array([]))
        orient_deg = float(np.degrees(np.arctan2(eigvecs[0, 1], eigvecs[0, 0])))

    return {
        'length_px':    skel_px,
        'orientation_deg': round(orient_deg, 2),
        'area_px':      area_px,
        'avg_width_px': round(avg_w, 3),
        'mask_found':   True,
    }


print('mask_to_geometry() defined ✓')

In [ ]:
# ── Build labels.csv ──────────────────────────────────────────────────────────

def resolution_to_str(r) -> str:
    """0.5 → '0.5',  1.0 → '1'  (matches actual filename convention)"""
    return '1' if float(r) == 1.0 else str(float(r))


def find_image(sample: str, res_str: str) -> Optional[str]:
    """Locate the .JPG file, tolerating minor naming variations."""
    candidates = [
        Path(IMAGES_DIR) / sample / f'{sample}_{res_str}_cropped.JPG',
        Path(IMAGES_DIR) / sample / f'{sample}_{res_str}_cropped.jpg',
        Path(IMAGES_DIR) / sample / f'{sample}_{res_str}.JPG',
        Path(IMAGES_DIR) / sample / f'{sample}_{res_str}.jpg',
    ]
    for p in candidates:
        if p.exists(): return str(p)
    return None


def find_mask(sample: str, res_str: str) -> Optional[str]:
    """Locate the .PNG mask, tolerating minor naming variations."""
    candidates = [
        Path(MASKS_DIR) / sample / f'{sample}_{res_str}_cropped.PNG',
        Path(MASKS_DIR) / sample / f'{sample}_{res_str}_cropped.png',
        Path(MASKS_DIR) / sample / f'{sample}_{res_str}.PNG',
        Path(MASKS_DIR) / sample / f'{sample}_{res_str}.png',
    ]
    for p in candidates:
        if p.exists(): return str(p)
    return None


# Load CSV
roi_df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
roi_df.columns = roi_df.columns.str.strip()

# Aggregate 5 ROI widths per image → mean & max
agg = roi_df.groupby(['Sample', 'Resolution'])['Crack Ruler Measurement'].agg(
    mean_width_mm='mean',
    max_width_mm='max',
    min_width_mm='min',
    std_width_mm='std',
).reset_index()

records = []
missing_imgs, missing_masks = [], []

for _, row in agg.iterrows():
    sample  = row['Sample']
    res_str = resolution_to_str(row['Resolution'])

    img_path  = find_image(sample, res_str)
    mask_path = find_mask(sample, res_str)

    if img_path is None:
        missing_imgs.append(f'{sample}_{res_str}')
        continue

    # Geometry from mask
    geom = mask_to_geometry(mask_path) if mask_path else \
           {'length_px': 0, 'orientation_deg': 0.0,
            'area_px': 0, 'avg_width_px': 0.0, 'mask_found': False}

    if mask_path is None:
        missing_masks.append(f'{sample}_{res_str}')

    records.append({
        'filename':       img_path,
        'sample':         sample,
        'resolution':     float(row['Resolution']),
        # ── CNN regression targets ────────────────────────────────────────
        'mean_width_mm':  round(float(row['mean_width_mm']), 4),
        'max_width_mm':   round(float(row['max_width_mm']),  4),
        'length_px':      geom['length_px'],
        'orientation_deg':geom['orientation_deg'],
        # ── Extra info ───────────────────────────────────────────────────
        'area_px':        geom['area_px'],
        'avg_width_px':   geom['avg_width_px'],
        'mask_path':      mask_path or '',
        'mask_found':     geom['mask_found'],
    })

labels_df = pd.DataFrame(records)
labels_df.to_csv(LABELS_CSV, index=False)

print(f'labels.csv  →  {LABELS_CSV}')
print(f'Total rows  :  {len(labels_df)}')
print(f'Masks found :  {labels_df["mask_found"].sum()} / {len(labels_df)}')
if missing_imgs:  print(f'Missing images : {missing_imgs}')
if missing_masks: print(f'Missing masks  : {missing_masks}')
print()
print(labels_df[['sample','resolution','mean_width_mm','max_width_mm',
                  'length_px','orientation_deg']].head(9).to_string(index=False))

In [ ]:
# ── Explore label distributions ───────────────────────────────────────────────
TARGET_COLS = ['mean_width_mm', 'max_width_mm', 'length_px', 'orientation_deg']

print('Label statistics:')
print(labels_df[TARGET_COLS].describe().round(3))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Target Label Distributions', color='white', fontsize=14)
colors = [C_BLUE, C_GREEN, C_ORANGE, C_PURPLE]
units  = ['mm', 'mm', 'px', '°']

for ax, col, clr, unit in zip(axes.flat, TARGET_COLS, colors, units):
    ax.hist(labels_df[col], bins=15, color=clr, alpha=0.85, edgecolor='none')
    ax.axvline(labels_df[col].mean(), color='white', lw=1.5, linestyle='--', label='mean')
    ax.set_title(f'{col}  ({unit})', color=clr)
    ax.set_xlabel(f'Value ({unit})')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/label_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visualise a few images with their mask overlays ───────────────────────────
n_show  = min(6, len(labels_df))
samples = labels_df.sample(n_show, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle('Images + Ground Truth Masks + Labels', color='white', fontsize=13)

for ax, (_, row) in zip(axes.flat, samples.iterrows()):
    img = np.array(Image.open(row['filename']).convert('RGB'))

    # Overlay mask in red if available
    if row['mask_found'] and row['mask_path']:
        mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
        _, mask_bin = cv2.threshold(mask, 10, 1, cv2.THRESH_BINARY)
        overlay = img.copy()
        overlay[mask_bin == 1] = [255, 50, 50]
        img = cv2.addWeighted(img, 0.65, overlay, 0.35, 0)

    ax.imshow(img)
    title = (f"{row['sample']}  @{row['resolution']}m\n"
             f"mean_w={row['mean_width_mm']:.2f}mm  "
             f"max_w={row['max_width_mm']:.2f}mm\n"
             f"len={row['length_px']}px  θ={row['orientation_deg']:.1f}°")
    ax.set_title(title, fontsize=7, color=C_GREEN)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## 3 · Dataset class & Data splits

**Small dataset strategy (45 images):**
- Split by **physical crack** (Sample_X), NOT randomly by image
  → prevents data leakage (same crack at 3 distances would appear in both train & test)
- Heavy augmentation during training
- Frozen ResNet50 backbone (only last layer + heads trainable)

In [ ]:
# ── Compute normalisation stats from training targets ─────────────────────────

# Split by SAMPLE (15 physical cracks) to avoid leakage
all_samples = sorted(labels_df['sample'].unique())   # 15 unique cracks
n_test  = max(1, round(len(all_samples) * TEST_SPLIT))
n_val   = max(1, round(len(all_samples) * VAL_SPLIT))
n_train = len(all_samples) - n_test - n_val

rng           = np.random.default_rng(RANDOM_SEED)
shuffled      = rng.permutation(all_samples).tolist()
train_samples = shuffled[:n_train]
val_samples   = shuffled[n_train:n_train + n_val]
test_samples  = shuffled[n_train + n_val:]

train_df = labels_df[labels_df['sample'].isin(train_samples)].reset_index(drop=True)
val_df   = labels_df[labels_df['sample'].isin(val_samples)].reset_index(drop=True)
test_df  = labels_df[labels_df['sample'].isin(test_samples)].reset_index(drop=True)

print(f'Train : {len(train_df)} images  ({n_train} cracks: {train_samples})')
print(f'Val   : {len(val_df)} images  ({n_val} cracks: {val_samples})')
print(f'Test  : {len(test_df)} images  ({len(test_samples)} cracks: {test_samples})')

# Compute normalisation stats from train set only
LABEL_STATS = {}
for col in TARGET_COLS:
    LABEL_STATS[col] = {
        'mean': float(train_df[col].mean()),
        'std':  float(train_df[col].std()) + 1e-8,
    }

print('\nLabel normalisation stats (from train set):')
for k, v in LABEL_STATS.items():
    print(f'  {k:<20}: mean={v["mean"]:8.3f}  std={v["std"]:8.3f}')

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────

def get_transforms(split: str):
    norm = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    if split == 'train':
        # Heavy augmentation — critical for only 27 training images
        return T.Compose([
            T.Resize((IMG_SIZE + 64, IMG_SIZE + 64)),
            T.RandomCrop(IMG_SIZE),
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.3),
            T.RandomRotation(45),
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
            T.RandomGrayscale(0.15),
            T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
            T.RandomErasing(p=0.2, scale=(0.02, 0.1)),   # simulate occlusion
            T.ToTensor(), norm,
        ])
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), norm])


# ── Dataset class ─────────────────────────────────────────────────────────────

class CrackDataset(Dataset):
    """
    Returns (image_tensor, normalised_label_tensor, raw_label_tensor, filename)
    """
    def __init__(self, df: pd.DataFrame, split: str = 'train'):
        self.df        = df.reset_index(drop=True)
        self.transform = get_transforms(split)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(row['filename']).convert('RGB')
        img  = self.transform(img)
        raw  = torch.tensor([row[c] for c in TARGET_COLS], dtype=torch.float32)
        norm = self._normalise(raw)
        return img, norm, raw, row['filename']

    def _normalise(self, t: torch.Tensor) -> torch.Tensor:
        out = t.clone()
        for i, c in enumerate(TARGET_COLS):
            out[i] = (t[i] - LABEL_STATS[c]['mean']) / LABEL_STATS[c]['std']
        return out

    @staticmethod
    def denormalise(t: torch.Tensor) -> torch.Tensor:
        out = t.clone()
        for i, c in enumerate(TARGET_COLS):
            out[..., i] = t[..., i] * LABEL_STATS[c]['std'] + LABEL_STATS[c]['mean']
        return out


train_ds = CrackDataset(train_df, 'train')
val_ds   = CrackDataset(val_df,   'val')
test_ds  = CrackDataset(test_df,  'test')

# Drop-last=False because test/val sets are tiny
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'DataLoaders ready.  Train batches: {len(train_loader)}')

## 4 · CNN Model

In [ ]:
# ── Architecture ──────────────────────────────────────────────────────────────

class RegressionHead(nn.Module):
    """Dense regression head with Softplus to enforce output > 0."""
    def __init__(self, in_f=512, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(hidden, 64), nn.GELU(),
            nn.Linear(64, 1), nn.Softplus(),
        )
    def forward(self, x): return self.net(x).squeeze(-1)


class OrientationHead(nn.Module):
    """Predicts (sin θ, cos θ) → degrees.  Avoids angle-wrap discontinuity."""
    def __init__(self, in_f=512, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, 2),
        )
    def forward(self, x):
        sc = self.net(x)
        return torch.rad2deg(torch.atan2(sc[:, 0], sc[:, 1]))


class DamCrackCNN(nn.Module):
    """
    ResNet50 backbone (ImageNet pretrained, backbone frozen except layer4)
    + shared 512-dim layer
    + four heads: mean_width | max_width | length | orientation

    For 45 images, freezing most of the backbone is essential to avoid overfitting.
    """
    def __init__(self, pretrained=True):
        super().__init__()
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        in_f = backbone.fc.in_features   # 2048
        backbone.fc = nn.Identity()

        # Freeze everything except layer4 + layer3 (gradual unfreezing)
        for p in backbone.parameters(): p.requires_grad = False
        for p in backbone.layer4.parameters(): p.requires_grad = True
        for p in backbone.layer3.parameters(): p.requires_grad = True

        self.backbone = backbone
        self.shared   = nn.Sequential(
            nn.Linear(in_f, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.5)
        )
        self.head_mean_width  = RegressionHead()
        self.head_max_width   = RegressionHead()
        self.head_length      = RegressionHead()
        self.head_orientation = OrientationHead()

    def forward(self, x):
        f = self.shared(self.backbone(x))
        return {
            'mean_width':  self.head_mean_width(f),
            'max_width':   self.head_max_width(f),
            'length':      self.head_length(f),
            'orientation': self.head_orientation(f),
        }


model = DamCrackCNN(pretrained=True).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:>12,}')
print(f'Trainable params: {trainable:>12,}')
print(f'Frozen params   : {total - trainable:>12,}')

# Sanity check
imgs_t, _, _, _ = next(iter(train_loader))
with torch.no_grad():
    out = model(imgs_t[:2].to(DEVICE))
print('\nOutput shapes:')
for k, v in out.items(): print(f'  {k:<16}: {v.shape}')

## 5 · Training

In [ ]:
# ── Loss & optimiser ──────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """
    Weighted MSE across 4 heads.
    Width heads get higher weight — they are the primary RUL input.
    Weights: [mean_width, max_width, length, orientation]
    """
    def __init__(self, weights=(2.0, 1.5, 0.5, 0.3)):
        super().__init__()
        self.register_buffer('w', torch.tensor(weights))
        self.mse = nn.MSELoss()

    def forward(self, preds: dict, targets: torch.Tensor):
        keys   = ['mean_width', 'max_width', 'length', 'orientation']
        losses = torch.stack([self.mse(preds[k], targets[:, i]) for i, k in enumerate(keys)])
        return (losses * self.w).sum(), dict(zip(keys, losses.detach().cpu().tolist()))


criterion = MultiTaskLoss()

# Differential LR: backbone unfrozen layers get 10× lower LR
backbone_params = list(model.backbone.layer3.parameters()) + \
                  list(model.backbone.layer4.parameters())
head_params     = list(model.shared.parameters()) + \
                  list(model.head_mean_width.parameters()) + \
                  list(model.head_max_width.parameters()) + \
                  list(model.head_length.parameters()) + \
                  list(model.head_orientation.parameters())

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': LR * 0.1},
    {'params': head_params,     'lr': LR},
], weight_decay=1e-3)

# CosineAnnealingLR works better than OneCycleLR for very small datasets
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=1e-7
)

scaler = GradScaler() if torch.cuda.is_available() else None
print('Loss, optimiser, scheduler ready ✓')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────

def run_epoch(loader, train: bool):
    model.train(train)
    tot_loss = 0.0
    tot_mae  = {k: 0.0 for k in ['mean_width','max_width','length','orientation']}
    n = 0
    for batch in loader:
        imgs_b, lbl_norm, _, _ = batch
        imgs_b   = imgs_b.to(DEVICE)
        lbl_norm = lbl_norm.to(DEVICE)
        with autocast(enabled=(scaler is not None)):
            preds = model(imgs_b)
            loss, _ = criterion(preds, lbl_norm)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        tot_loss += loss.item()
        for i, k in enumerate(['mean_width','max_width','length','orientation']):
            tot_mae[k] += (preds[k] - lbl_norm[:, i]).abs().mean().item()
        n += 1
    return tot_loss / max(n,1), {k: v / max(n,1) for k, v in tot_mae.items()}


log_path     = Path(SAVE_DIR) / 'training_log.csv'
best_val     = float('inf')
patience_cnt = 0
history      = []

with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow([
        'epoch','train_loss','val_loss',
        'tr_mean_w','tr_max_w','tr_len','tr_ori',
        'va_mean_w','va_max_w','va_len','va_ori','lr','t_s'
    ])

print(f'Training on {DEVICE}  |  {len(train_ds)} train images  |  up to {EPOCHS} epochs\n')
print(f'{"Ep":>4}  {"TrainLoss":>10}  {"ValLoss":>10}  '
      f'{"mW-MAE":>8}  {"xW-MAE":>8}  {"Len-MAE":>8}  {"Ori-MAE":>8}  {"LR":>9}')
print('─' * 85)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_mae = run_epoch(train_loader, train=True)
    scheduler.step()
    with torch.no_grad():
        va_loss, va_mae = run_epoch(val_loader, train=False)
    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[-1]['lr']

    print(f'{epoch:4d}  {tr_loss:10.4f}  {va_loss:10.4f}  '
          f'{va_mae["mean_width"]:8.4f}  {va_mae["max_width"]:8.4f}  '
          f'{va_mae["length"]:8.4f}  {va_mae["orientation"]:8.4f}  {lr_now:9.2e}')

    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch, tr_loss, va_loss,
            tr_mae['mean_width'], tr_mae['max_width'], tr_mae['length'], tr_mae['orientation'],
            va_mae['mean_width'], va_mae['max_width'], va_mae['length'], va_mae['orientation'],
            lr_now, round(elapsed, 2)
        ])

    history.append({'epoch': epoch, 'train': tr_loss, 'val': va_loss})

    if va_loss < best_val:
        best_val, patience_cnt = va_loss, 0
        torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                    'val_loss': va_loss, 'label_stats': LABEL_STATS}, BEST_MODEL)
        print(f'  ✓ Best model saved  (val={va_loss:.4f})')
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            break

print(f'\nDone.  Best val loss = {best_val:.4f}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
df_log = pd.read_csv(log_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Progress', color='white', fontsize=14)

ax = axes[0]
ax.plot(df_log['epoch'], df_log['train_loss'], color=C_BLUE,   lw=2, label='Train')
ax.plot(df_log['epoch'], df_log['val_loss'],   color=C_ORANGE, lw=2, linestyle='--', label='Val')
ax.set_title('Multi-task Loss', color=C_BLUE)
ax.set_xlabel('Epoch'); ax.legend()

ax = axes[1]
for col, lbl, clr in [('va_mean_w','mean width',C_BLUE),('va_max_w','max width',C_GREEN),
                       ('va_len','length',C_ORANGE),('va_ori','orientation',C_PURPLE)]:
    ax.plot(df_log['epoch'], df_log[col], color=clr, lw=2, label=lbl)
ax.set_title('Val MAE (normalised)', color=C_GREEN)
ax.set_xlabel('Epoch'); ax.legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 6 · Inference & Evaluation on Test Set

In [ ]:
# ── Load best model & define predict() ────────────────────────────────────────
ckpt = torch.load(BEST_MODEL, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
# Restore label stats from checkpoint (important for deployment)
LABEL_STATS = ckpt.get('label_stats', LABEL_STATS)
print(f'Loaded epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')

INFER_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


def predict(img_path: str) -> dict:
    """
    Predict crack dimensions from a single image file.
    Returns dict with physical-unit predictions.
    """
    pil = Image.open(img_path).convert('RGB')
    t   = INFER_TF(pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        raw = model(t)
    # Stack normalised outputs then denormalise
    pred_norm = torch.stack([
        raw['mean_width'], raw['max_width'],
        raw['length'],     raw['orientation']
    ], dim=1)[0].cpu()
    result = {}
    for i, c in enumerate(TARGET_COLS):
        phys = pred_norm[i].item() * LABEL_STATS[c]['std'] + LABEL_STATS[c]['mean']
        result[c] = round(float(phys), 4)
    return result


print('predict() ready ✓')

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────────────
preds_all, trues_all, fnames = [], [], []

for _, row in test_df.iterrows():
    pred = predict(row['filename'])
    preds_all.append([pred[c] for c in TARGET_COLS])
    trues_all.append([row[c]  for c in TARGET_COLS])
    fnames.append(row['filename'])

P = np.array(preds_all)
T = np.array(trues_all)
mae_per  = np.abs(P - T).mean(axis=0)
rmse_per = np.sqrt(((P - T)**2).mean(axis=0))

units_map = {'mean_width_mm':'mm','max_width_mm':'mm','length_px':'px','orientation_deg':'°'}

print(f'Test set metrics  ({len(test_df)} images):')
print(f'{"Target":<22} {"MAE":>10} {"RMSE":>10}  Unit')
print('─'*50)
for c, mae, rmse in zip(TARGET_COLS, mae_per, rmse_per):
    print(f'{c:<22} {mae:10.4f} {rmse:10.4f}  {units_map[c]}')

In [ ]:
# ── Predicted vs True scatter ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Predicted vs True — Test Set', color='white', fontsize=14)
colors = [C_BLUE, C_GREEN, C_ORANGE, C_PURPLE]

for ax, c, clr in zip(axes.flat, TARGET_COLS, colors):
    idx = TARGET_COLS.index(c)
    tr, pr = T[:,idx], P[:,idx]
    lim = [min(tr.min(), pr.min())*0.9, max(tr.max(), pr.max())*1.1]
    ax.scatter(tr, pr, color=clr, alpha=0.8, s=60, edgecolors='white', lw=0.3)
    ax.plot(lim, lim, 'w--', lw=1, alpha=0.6)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f'True {c}')
    ax.set_ylabel(f'Predicted {c}')
    ax.set_title(f'{c}\nMAE={mae_per[idx]:.4f}  RMSE={rmse_per[idx]:.4f}',
                 color=clr, fontsize=9)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/pred_vs_true.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Visual inspection: image + predictions vs ground truth ───────────────────
n_show = min(6, len(test_df))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Test Images  |  Predicted (cyan)  vs  True (orange)', color='white', fontsize=12)

for ax, (_, row) in zip(axes.flat, test_df.sample(n_show, random_state=0).iterrows()):
    pred = predict(row['filename'])
    img  = np.array(Image.open(row['filename']).convert('RGB'))

    # Overlay mask
    if row['mask_found'] and row['mask_path']:
        mask = cv2.imread(row['mask_path'], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]))
        _, mb = cv2.threshold(mask, 10, 1, cv2.THRESH_BINARY)
        ov = img.copy(); ov[mb==1] = [255,80,80]
        img = cv2.addWeighted(img, 0.6, ov, 0.4, 0)

    ax.imshow(img)
    ax.text(0.02, 0.98,
            f"PRED\nmean_w={pred['mean_width_mm']:.3f}mm\n"
            f"max_w={pred['max_width_mm']:.3f}mm\n"
            f"len={pred['length_px']:.0f}px\nθ={pred['orientation_deg']:.1f}°",
            transform=ax.transAxes, va='top', fontsize=7, color=C_BLUE,
            bbox=dict(fc='black', alpha=0.65, pad=2))
    ax.text(0.55, 0.98,
            f"TRUE\nmean_w={row['mean_width_mm']:.3f}mm\n"
            f"max_w={row['max_width_mm']:.3f}mm\n"
            f"len={row['length_px']}px\nθ={row['orientation_deg']:.1f}°",
            transform=ax.transAxes, va='top', fontsize=7, color=C_ORANGE,
            bbox=dict(fc='black', alpha=0.65, pad=2))
    name = Path(row['filename']).name
    ax.set_title(f"{name}  @{row['resolution']}m", fontsize=7, color=C_GREY)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/visual_inspection.png', dpi=120, bbox_inches='tight')
plt.show()

## 7 · Fracture Mechanics & RUL Engine

**Physics chain (from your PDF):**

$$p(y) = \rho g y \quad\rightarrow\quad \sigma = \frac{p}{t} \quad\rightarrow\quad \sigma_n = \sigma\cos^2\theta$$

$$K_I = Y\,\sigma_n\sqrt{\pi a} \quad [\text{MPa}\cdot\sqrt{\text{m}}]$$

$$\frac{da}{dt} = C\,(K_I)^m \quad\rightarrow\quad \text{RUL} = \sum \Delta t \text{ until } K_I \geq K_{IC}$$

The **crack depth `a`** fed into the SIF formula is the max crack width (in metres)  
— this is the most directly measured dimension and represents the crack's penetration depth  
into the dam face, which governs fracture.

In [ ]:
# ── Physics engine ────────────────────────────────────────────────────────────

@dataclass
class DamParams:
    H:   float = DAM_H;    t:   float = DAM_T
    rho: float = 1000.0;   g:   float = 9.81
    KIC: float = KIC;      Y:   float = Y_GEOM
    C:   float = PARIS_C;  m:   float = PARIS_M
    a_critical: Optional[float] = None


@dataclass
class RULResult:
    rul_years:      float
    rul_capped:     bool          # True = safely exceeds simulation horizon
    current_KI:     float         # MPa·√m
    ratio_KI_KIC:   float         # primary health index
    a_initial_mm:   float         # mm — crack depth (= max_width_mm)
    a_critical_mm:  float         # mm
    sigma_n_MPa:    float
    risk_level:     str
    growth_history: list = field(default_factory=list)  # (t_yr, a_mm, KI)


class RULEngine:
    SPY = 365.25 * 24 * 3600

    def __init__(self, dam: DamParams, dt_s=86400.0, max_years=50000.0, store_every=365):
        self.dam         = dam
        self.dt          = dt_s
        self.max_steps   = int(max_years * self.SPY / dt_s)
        self.store_every = store_every

    def compute(self, cnn_pred: dict) -> RULResult:
        """
        cnn_pred: dict with keys from TARGET_COLS
          mean_width_mm, max_width_mm, length_px, orientation_deg

        Crack depth 'a' = max_width_mm converted to metres.
        Water depth y   = conservative: 40% of dam height.
        """
        dam   = self.dam
        theta = cnn_pred.get('orientation_deg', 0.0)

        # Crack depth in metres (max_width is most critical dimension)
        a_mm = max(cnn_pred.get('max_width_mm', 1.0), 0.01)
        a    = a_mm / 1000.0   # mm → m

        # Conservative water depth
        y = dam.H * 0.4

        # Stress chain (Pa → MPa)
        p       = dam.rho * dam.g * y
        sigma   = p / dam.t
        sigma_n = sigma * math.cos(math.radians(theta)) ** 2
        s_MPa   = sigma_n / 1e6

        # SIF in MPa·√m
        KI0 = dam.Y * s_MPa * math.sqrt(math.pi * a)

        # Critical crack depth
        if dam.a_critical:
            a_crit = dam.a_critical
        elif s_MPa > 0:
            a_crit = (dam.KIC / (dam.Y * s_MPa)) ** 2 / math.pi
        else:
            a_crit = float('inf')

        ratio = KI0 / dam.KIC
        risk  = self._risk(ratio)

        if a >= a_crit or KI0 >= dam.KIC:
            return RULResult(0.0, False, round(KI0,6), round(ratio,6),
                             round(a*1000,4), round(a_crit*1000,4),
                             round(s_MPa,6), 'CRITICAL — Immediate action')

        a_cur, t_s, step = a, 0.0, 0
        hist, capped = [], False

        while a_cur < a_crit and step < self.max_steps:
            KI    = dam.Y * s_MPa * math.sqrt(math.pi * a_cur)
            da_dt = dam.C * (KI ** dam.m)
            a_cur += da_dt * self.dt
            t_s   += self.dt; step += 1
            if self.store_every > 0 and step % self.store_every == 0:
                hist.append((round(t_s/self.SPY,3), round(a_cur*1000,5), round(KI,7)))

        if step >= self.max_steps:
            capped = True

        return RULResult(
            rul_years     = round(t_s / self.SPY, 3),
            rul_capped    = capped,
            current_KI    = round(KI0, 6),
            ratio_KI_KIC  = round(ratio, 6),
            a_initial_mm  = round(a*1000, 4),
            a_critical_mm = round(a_crit*1000, 4),
            sigma_n_MPa   = round(s_MPa, 6),
            risk_level    = risk,
            growth_history= hist,
        )

    @staticmethod
    def _risk(r):
        if r >= 1.0: return 'CRITICAL — Immediate action'
        if r >= 0.8: return 'HIGH — Urgent inspection'
        if r >= 0.6: return 'MODERATE — Close monitoring'
        if r >= 0.4: return 'ELEVATED — Routine monitoring'
        return               'LOW — Normal operation'


engine = RULEngine(DamParams(), dt_s=86400, max_years=50000, store_every=365)
print('RUL engine ready ✓')

# Quick demo with sample measurements from the dataset
demo = {'mean_width_mm': 2.29, 'max_width_mm': 4.0,
        'length_px': 800, 'orientation_deg': 10.0}
r = engine.compute(demo)
print(f'\nDemo (Sample_1 crack):')
print(f'  max_width = {demo["max_width_mm"]} mm  →  a = {r.a_initial_mm} mm')
print(f'  K_I = {r.current_KI:.5f} MPa√m   K_IC = {engine.dam.KIC} MPa√m')
print(f'  K_I/K_IC = {r.ratio_KI_KIC:.5f}')
print(f'  Risk: {r.risk_level}')
print(f'  RUL: {">50,000" if r.rul_capped else r.rul_years} years')

In [ ]:
# ── Compute RUL for the full test set ─────────────────────────────────────────
rul_records = []

for _, row in test_df.iterrows():
    pred = predict(row['filename'])
    res  = engine.compute(pred)
    rul_records.append({
        'filename':      Path(row['filename']).name,
        'sample':        row['sample'],
        'resolution':    row['resolution'],
        'pred_mean_w':   pred['mean_width_mm'],
        'pred_max_w':    pred['max_width_mm'],
        'true_mean_w':   row['mean_width_mm'],
        'true_max_w':    row['max_width_mm'],
        'KI_MPa':        res.current_KI,
        'KI_KIC':        res.ratio_KI_KIC,
        'sigma_n_MPa':   res.sigma_n_MPa,
        'a_crit_mm':     res.a_critical_mm,
        'RUL_yr':        res.rul_years,
        'RUL_capped':    res.rul_capped,
        'risk':          res.risk_level,
    })

df_rul = pd.DataFrame(rul_records)
df_rul.to_csv(f'{SAVE_DIR}/rul_results.csv', index=False)

print('RUL results (test set):')
print(df_rul[['filename','pred_max_w','KI_KIC','RUL_yr','RUL_capped','risk']].to_string(index=False))

## 8 · Dashboard & Visualisations

In [ ]:
# ── Compute RUL for ALL 45 images (full dataset view) ─────────────────────────
all_rul = []
for _, row in labels_df.iterrows():
    pred = predict(row['filename'])
    res  = engine.compute(pred)
    all_rul.append({
        'filename':   Path(row['filename']).name,
        'sample':     row['sample'],
        'resolution': row['resolution'],
        'pred_max_w': pred['max_width_mm'],
        'true_max_w': row['max_width_mm'],
        'KI_KIC':     res.ratio_KI_KIC,
        'RUL_yr':     res.rul_years,
        'RUL_capped': res.rul_capped,
        'risk':       res.risk_level,
    })

df_all = pd.DataFrame(all_rul)

# ── Dashboard ─────────────────────────────────────────────────────────────────
risk_clr = {
    'LOW — Normal operation':         C_GREEN,
    'ELEVATED — Routine monitoring':  '#AAFF00',
    'MODERATE — Close monitoring':    C_ORANGE,
    'HIGH — Urgent inspection':       '#FF8800',
    'CRITICAL — Immediate action':    C_RED,
}

fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#0D1117')
gs  = gridspec.GridSpec(2, 3, hspace=0.38, wspace=0.32)

# 1 — K_I/K_IC histogram
ax1 = fig.add_subplot(gs[0,0])
ax1.hist(df_all['KI_KIC'], bins=15, color=C_BLUE, alpha=0.85, edgecolor='none')
ax1.axvline(1.0, color=C_RED, lw=2, linestyle='--', label='Failure')
ax1.set_title('K_I / K_IC  (all 45 images)', color=C_BLUE)
ax1.set_xlabel('K_I / K_IC'); ax1.legend()

# 2 — Risk distribution
ax2 = fig.add_subplot(gs[0,1])
rc  = df_all['risk'].value_counts()
ax2.pie(rc.values, labels=rc.index, colors=[risk_clr.get(r,C_GREY) for r in rc.index],
        autopct='%1.0f%%', textprops={'fontsize':7,'color':'white'})
ax2.set_title('Risk Distribution', color=C_ORANGE)

# 3 — K_I/K_IC vs max_width (predicted)
ax3 = fig.add_subplot(gs[0,2])
sc  = ax3.scatter(df_all['pred_max_w'], df_all['KI_KIC'],
                  c=df_all['KI_KIC'], cmap='RdYlGn_r', vmin=0, vmax=0.5, s=35, alpha=0.85)
ax3.set_xlabel('Predicted max width (mm)')
ax3.set_ylabel('K_I / K_IC')
ax3.set_title('Width vs Health Index', color=C_GREEN)
plt.colorbar(sc, ax=ax3)

# 4 — Pred vs True width (all 45 images)
ax4 = fig.add_subplot(gs[1,0])
ax4.scatter(df_all['true_max_w'], df_all['pred_max_w'],
            color=C_BLUE, alpha=0.75, s=30, edgecolors='white', lw=0.3)
lim = [0, df_all[['true_max_w','pred_max_w']].values.max()*1.05]
ax4.plot(lim, lim, 'w--', lw=1)
ax4.set_xlabel('True max width (mm)'); ax4.set_ylabel('Predicted max width (mm)')
ax4.set_title('Width Prediction Quality', color=C_PURPLE)

# 5 — K_I/K_IC per sample (bar chart, average across 3 resolutions)
ax5 = fig.add_subplot(gs[1,1:])
per_sample = df_all.groupby('sample')['KI_KIC'].mean().sort_values(ascending=False)
bc = [risk_clr.get(engine._risk(v), C_GREY) for v in per_sample.values]
ax5.bar(per_sample.index, per_sample.values, color=bc, edgecolor='none')
ax5.axhline(1.0, color=C_RED, lw=1.5, linestyle='--', label='K_IC limit')
ax5.axhline(0.6, color=C_ORANGE, lw=1, linestyle=':', label='MODERATE threshold')
ax5.set_xticklabels(per_sample.index, rotation=45, ha='right', fontsize=8)
ax5.set_ylabel('Mean K_I / K_IC')
ax5.set_title('Health Index per Sample (averaged over 3 distances)', color=C_RED)
ax5.legend(fontsize=8)

plt.suptitle('Dam SHM — Structural Health Dashboard\n(VT Crack Quantification Dataset)',
             fontsize=14, color='white', y=1.01)
plt.savefig(f'{SAVE_DIR}/dashboard.png', dpi=130, bbox_inches='tight', facecolor='#0D1117')
plt.show()

In [ ]:
# ── Effect of standoff distance on predictions ────────────────────────────────
# The dataset has same crack at 3 distances — ideal to test scale invariance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Effect of Camera Standoff Distance on Predictions', color='white', fontsize=13)

for res, clr in [(0.5, C_BLUE), (0.75, C_GREEN), (1.0, C_ORANGE)]:
    sub = df_all[df_all['resolution'] == res]
    axes[0].scatter(sub['true_max_w'], sub['pred_max_w'],
                    color=clr, s=35, alpha=0.8, label=f'{res}m', edgecolors='none')
    axes[1].scatter(sub['true_max_w'], sub['KI_KIC'],
                    color=clr, s=35, alpha=0.8, label=f'{res}m', edgecolors='none')

axes[0].plot([0,10],[0,10],'w--',lw=1)
axes[0].set_xlabel('True max width (mm)'); axes[0].set_ylabel('Predicted max width (mm)')
axes[0].set_title('Width prediction by distance', color=C_BLUE); axes[0].legend()

axes[1].set_xlabel('True max width (mm)'); axes[1].set_ylabel('K_I / K_IC')
axes[1].set_title('Health index by distance', color=C_GREEN); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/scale_invariance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sensitivity analysis: RUL vs fracture toughness K_IC ─────────────────────
# Use the worst crack (highest K_I/K_IC) from the full dataset
worst_row  = df_all.loc[df_all['KI_KIC'].idxmax()]
worst_pred = predict(labels_df[labels_df['sample']==worst_row['sample']].iloc[0]['filename'])

print(f'Worst crack: {worst_row["sample"]}  K_I/K_IC={worst_row["KI_KIC"]:.5f}  Risk: {worst_row["risk"]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

kic_vals = np.linspace(0.5, 2.0, 30)
rul_kic  = []
for kv in kic_vals:
    r = RULEngine(DamParams(KIC=kv), dt_s=86400, max_years=50000).compute(worst_pred)
    rul_kic.append(min(r.rul_years, 50000))

axes[0].plot(kic_vals, rul_kic, color=C_BLUE, lw=2)
axes[0].fill_between(kic_vals, rul_kic, alpha=0.12, color=C_BLUE)
axes[0].axvline(KIC, color=C_ORANGE, lw=1.5, linestyle='--', label=f'Design K_IC={KIC}')
axes[0].set_xlabel('Fracture Toughness K_IC (MPa√m)')
axes[0].set_ylabel('RUL (years)'); axes[0].set_title('Sensitivity: K_IC', color=C_BLUE)
axes[0].legend()

# Vary water level (dam H)
h_vals  = np.linspace(20, 100, 30)
rul_h   = []
for hv in h_vals:
    r = RULEngine(DamParams(H=hv), dt_s=86400, max_years=50000).compute(worst_pred)
    rul_h.append(min(r.rul_years, 50000))

axes[1].plot(h_vals, rul_h, color=C_GREEN, lw=2)
axes[1].fill_between(h_vals, rul_h, alpha=0.12, color=C_GREEN)
axes[1].axvline(DAM_H, color=C_ORANGE, lw=1.5, linestyle='--', label=f'Design H={DAM_H}m')
axes[1].set_xlabel('Water Head H (m)')
axes[1].set_ylabel('RUL (years)'); axes[1].set_title('Sensitivity: Water Level', color=C_GREEN)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sensitivity_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## 9 · Single-image inspection  (use on any new photo)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  Change IMAGE_PATH to inspect any crack photo from the dam
# ══════════════════════════════════════════════════════════════════
IMAGE_PATH = 'path/to/new_crack_photo.JPG'   # ← EDIT THIS

# Fallback: use first test image
if not Path(IMAGE_PATH).exists():
    IMAGE_PATH = test_df['filename'].iloc[0]
    print(f'[Using test image: {Path(IMAGE_PATH).name}]')

# ── Predict + RUL ─────────────────────────────────────────────────
pred    = predict(IMAGE_PATH)
rul_res = engine.compute(pred)

# ── Report card ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0D1117')

axes[0].imshow(Image.open(IMAGE_PATH))
axes[0].set_title('Input Image', color=C_BLUE, fontsize=12)
axes[0].axis('off')

axes[1].set_facecolor('#0D1117')
axes[1].axis('off')

rc = risk_clr.get(rul_res.risk_level, C_GREY)
lines = [
    ('CNN PREDICTIONS', '', 'white', True),
    ('Mean width',    f"{pred['mean_width_mm']:.3f} mm",  C_BLUE,   False),
    ('Max width',     f"{pred['max_width_mm']:.3f} mm",   C_BLUE,   False),
    ('Crack length',  f"{pred['length_px']:.0f} px",      C_BLUE,   False),
    ('Orientation',   f"{pred['orientation_deg']:.1f} °", C_BLUE,   False),
    ('', '', 'white', False),
    ('FRACTURE MECHANICS', '', 'white', True),
    ('σ_n',           f"{rul_res.sigma_n_MPa:.5f} MPa",   C_GREEN,  False),
    ('K_I',           f"{rul_res.current_KI:.5f} MPa√m",  C_GREEN,  False),
    ('K_IC',          f"{KIC} MPa√m",                     C_GREEN,  False),
    ('K_I / K_IC',    f"{rul_res.ratio_KI_KIC:.5f}",      C_GREEN,  False),
    ('a (max width)', f"{rul_res.a_initial_mm:.4f} mm",   C_GREEN,  False),
    ('a_critical',    f"{rul_res.a_critical_mm:.2f} mm",  C_GREEN,  False),
    ('', '', 'white', False),
    ('ASSESSMENT', '', 'white', True),
    ('RUL', f"{'> 50,000' if rul_res.rul_capped else rul_res.rul_years:,} years", rc, False),
    ('Risk level', rul_res.risk_level, rc, False),
]

y = 0.97
for label, value, clr, bold in lines:
    if label == '':
        y -= 0.03; continue
    fw = 'bold' if bold else 'normal'
    axes[1].text(0.03, y, label + ('' if bold else ':'),
                 transform=axes[1].transAxes, va='top',
                 fontsize=10 if bold else 9, color=clr, fontweight=fw)
    if value:
        axes[1].text(0.55, y, value, transform=axes[1].transAxes,
                     va='top', fontsize=9, color=clr, fontweight='bold')
    y -= 0.065

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/inspection_report.png', dpi=130, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print(f'Report saved → {SAVE_DIR}/inspection_report.png')